In [ ]:
using Pkg
Pkg.activate("..")
using Revise

In [ ]:
using bslLD, Plots, Statistics
bslLD.greet()

# bslLD.use_cuda!()

In [ ]:
grid =  bslLD.Grid([0.0,-4.0],[2*pi,4.0],[128,129],0.01,10000,1, 0.0, 1)

# initFuncv(v)= exp(-(v+2)^2 / 2) / sqrt(2*pi)+ exp(-(v-2)^2 / 2) / sqrt(2*pi)
initFuncv(v) = exp(-v^2 / 2) / sqrt(2*pi)

f = bslLD.Distribution(grid, 0.0001,initFuncv=initFuncv);
e = bslLD.empty_vectorfield(grid);

In [ ]:
mutable struct Diag
    rho::Vector
    f::Vector
    Ex::Vector
end
Diag() = Diag([], [], [])

function diags!(diags, f, rho, Ex, grid)
    grid.index[1] % 10 == 0 || return
    push!(diags.f, copy(f.data .- mean(f.data, dims=1)))
    push!(diags.rho, copy(rho.data[:]))
    push!(diags.Ex, copy(Ex.data[:]))
end

function step!(f, grid, diags)
    bslLD.advectX!(f, grid)
    rho = bslLD.compute_density(f, grid)
    sol = bslLD.solve_fields(bslLD.Moments(rho), grid, bslLD.AdiabaticFieldSolver())
    bslLD.advectV!(f, grid, sol.E)
    diags!(diags, f, rho, sol.E[1], grid)
end


In [ ]:
diags = Diag()
for i in grid.itime
    grid.index[1] = i
    step!(f, grid, diags)
end


In [ ]:
plot(grid.xaxes[1],diags.Ex[1], label="Ex")

In [ ]:

plot(grid.xaxes[1],diags.rho, label="rho")

In [ ]:
num_frames = size(diags.f, 1)
frames_to_plot = 1:num_frames

animation = @animate for i in frames_to_plot
    heatmap(transpose(Array(diags.f[i])),
        title = "Frame $i",
        xlabel = "x",
        ylabel = "v",
    )

end
gif(animation, "fdiag_heatmap_animation.gif", fps = 10)

In [ ]:
plot(map(x->(mean((x.-mean(x)).^2)), Array.(diags.rho)), yscale=:log10)